<a href="https://colab.research.google.com/github/cowbay878787/MachineLearning/blob/main/0709_Colab_LINE_Bot_with_GEMINI_Rag_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 4.4 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://twelve-sandfish-afraid.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://twelve-sandfish-afraid.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於台灣新竹縣新豐鄉的私立科技大學。學校秉持《大學》「在明明德，在新民，在止於至善」的精義，以「堅毅、求新、創造」為校訓，致力於培養具備高尚品德、專業學問與優良技術的全人人才。

**歷史沿革**
明新科技大學的歷史可追溯至1966年3月1日創校的「明新工業專科學校」，初期設有機械、土木、工業管理三科。1993年更名為「明新工商專科學校」。1997年7月，奉教育部核准改制為「明新技術學院」並附設專科部。2002年9月，學校正式升格為「明新科技大學」。2018年12月，更名為「明新學校財團法人明新科技大學」，邁向培育「跨域整合、務實創新與全人學習」專業人才的一流產業大學。

**地理位置與校園**
明新科技大學佔地逾三十公頃，地處新竹縣新豐鄉，依傍省道縱貫線，毗鄰中山高速公路，交通便利。其地理位置優越，鄰近新竹科學園區、新竹工業區及竹北生醫園區，享有豐富的產學合作資源。

**學術單位與特色**
目前，明新科技大學設有半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院等六個學院，涵蓋20個學系、2個學位學程（含1個博士學位學程）及11個碩士班。學校特別注重與產業接軌，定位為「一流產業大學」。

明新科大積極發展以下四大育才特色（MUST）：多元學習（Multidisciplinary Learning）、全球視野（Universal Perspective）、永續經營（Sustainable Operations）與技術創新（Technological Innovation），引導學生跨域學習，以成為產業搶手人才。其半導體學院更跨足國際，計畫與美國、日本、馬來西亞、澳洲及越南等國家的學術單位合作半導體國際人才培育計畫。

**辦學聲譽**
明新科技大學在產學合作及畢業生就業方面表現優異，曾榮獲《Cheers》雜誌「企業最愛大學生」調查中，全國私立科技大學第三名，北台灣技專校院第一名。根據1111人力銀行統計，半導體產業界最愛聘用的畢業生中，明新科大名列第四，是唯一入榜的私立科大，顯示其畢業生深受業界肯定。此外，學校也積極推動「智慧生活」的創新服務，並整合資源建置「永續智慧商務」的教學與實習場域，透過AI與生成式AI的導入，

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"U43c52fd7c75fca7cc0d739ac44075aad","events":[{"type":"message","message":{"type":"text","id":"616970199648174613","quoteToken":"h51CyO8ASP8O4jNdShcFui6LKa5CptJtTbxnsPlMn4kCXoIETrtXRu_jM5ILQ3c5bXnY7PjRGGGG7g612A0KmKON2UyWhnC5tsOwRjSEVCa5E7rdDoRppdrSH6bYvLvzBZ1MBeOe7isWwEqRxtbaTQ","markAsReadToken":"Pfw7BvsMER4GVsFoyJGtcHaSlDzkuVg_cqq-RBrZITkaEewNp10IoswQgF91W7FhUoHfqQU9vXKNvltTzV0_xWn-ztWqDJ1KhdRMkoufcs8fJT-gl5Mybnle719pF-T5arUB8TR9AaMmxreoBmjBubTtLvbMPhIygihXEPYfKA-76OAwyah46jVSkGaiJOZNiOyxUmwy5_m761eB7kXe8w","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT97W8MSRHK0CMMQ70VEY1QP","deliveryContext":{"isRedelivery":false},"timestamp":1780574199986,"source":{"type":"user","userId":"U51821de656e781ae06afc45a00816099"},"replyToken":"49cb9e5346d24072bbea

BODY:  {"destination":"U43c52fd7c75fca7cc0d739ac44075aad","events":[{"type":"message","message":{"type":"text","id":"616970199648174613","quoteToken":"h51CyO8ASP8O4jNdShcFui6LKa5CptJtTbxnsPlMn4kCXoIETrtXRu_jM5ILQ3c5bXnY7PjRGGGG7g612A0KmKON2UyWhnC5tsOwRjSEVCa5E7rdDoRppdrSH6bYvLvzBZ1MBeOe7isWwEqRxtbaTQ","markAsReadToken":"Pfw7BvsMER4GVsFoyJGtcHaSlDzkuVg_cqq-RBrZITkaEewNp10IoswQgF91W7FhUoHfqQU9vXKNvltTzV0_xWn-ztWqDJ1KhdRMkoufcs8fJT-gl5Mybnle719pF-T5arUB8TR9AaMmxreoBmjBubTtLvbMPhIygihXEPYfKA-76OAwyah46jVSkGaiJOZNiOyxUmwy5_m761eB7kXe8w","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT97W8MSRHK0CMMQ70VEY1QP","deliveryContext":{"isRedelivery":false},"timestamp":1780574199986,"source":{"type":"user","userId":"U51821de656e781ae06afc45a00816099"},"replyToken":"49cb9e5346d24072bbea42f6efa426c7","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 11:56:42] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"U43c52fd7c75fca7cc0d739ac44075aad","events":[{"type":"message","message":{"type":"file","id":"616970289909858526","markAsReadToken":"qqP3BCLgDkNcJ6-nMYtg4cPv9qMdP_dolfraacnmBVITMY8vSBLgs8_bwttWwZAmPyedKC-gdohn01Ti3EtK8pWY447JVUucNaK0HY-y9JdbS7dSjbdd7dvLc-R1RFm0OgH2T5hX7Cu7Kb5slIrYafOlgDJqbWqh3PEuTVzJjatxKL2_ZR1HhJV_bAUIUA6YG3p6Uy7DQ8kn0FCBqew-LA","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KT97XWV4FA5N27RST31VHF6Z","deliveryContext":{"isRedelivery":false},"timestamp":1780574253749,"source":{"type":"user","userId":"U51821de656e781ae06afc45a00816099"},"replyToken":"fffc836a650949f294301d29f3d0f32d","mode":"active"}]}


BODY:  {"destination":"U43c52fd7c75fca7cc0d739ac44075aad","events":[{"type":"message","message":{"type":"file","id":"616970289909858526","markAsReadToken":"qqP3BCLgDkNcJ6-nMYtg4cPv9qMdP_dolfraacnmBVITMY8vSBLgs8_bwttWwZAmPyedKC-gdohn01Ti3EtK8pWY447JVUucNaK0HY-y9JdbS7dSjbdd7dvLc-R1RFm0OgH2T5hX7Cu7Kb5slIrYafOlgDJqbWqh3PEuTVzJjatxKL2_ZR1HhJV_bAUIUA6YG3p6Uy7DQ8kn0FCBqew-LA","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KT97XWV4FA5N27RST31VHF6Z","deliveryContext":{"isRedelivery":false},"timestamp":1780574253749,"source":{"type":"user","userId":"U51821de656e781ae06afc45a00816099"},"replyToken":"fffc836a650949f294301d29f3d0f32d","mode":"active"}]}
檔案已下載：/content/uploaded_files/RAG.txt


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 11:57:38] "POST / HTTP/1.1" 200 -


檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/x3yh6hm32vuw


INFO:__main__:Request body: {"destination":"U43c52fd7c75fca7cc0d739ac44075aad","events":[{"type":"message","message":{"type":"text","id":"616970322256331230","quoteToken":"7k0tRRG6JOS6Dbtk7-dLuZsGHFBnEZIUL6I6W1ej3H7GmtEfqM1vvgwYJggvrB4aw6PnWcsOxt0R6NgtcTN2VcjUciVwZ027bKm2Lq9mbnCgiDRWTP3QwGhv4OjIv5r8GOKvMjM_3fDKQ9qgJCkryw","markAsReadToken":"o17wB9mPc807nBsI7JP4SECMka8nnRkDQBjjfLf3yhZ08FFs-1htmWYGadS3Du6fPPXnNUwRAHztcrFCylYcK2--sOym4GyGrLFBAhVpgvDLLfFlrtf4CY5a08gZRQMRsIgvdLT-A4FI4EvroN_Gfc6prAS_zArXcK6VUSj9fZ2Ws1BXvdm3KBoNeJWF6CBEkAwBjYHragA23BU4fKguNg","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT97YFWVT9W7146AKF4DE830","deliveryContext":{"isRedelivery":false},"timestamp":1780574273000,"source":{"type":"user","userId":"U51821de656e781ae06afc45a00816099"},"replyToken":"c8bdd95808af46ab99ef838e2bf29d8c","mode":"active"}]}


BODY:  {"destination":"U43c52fd7c75fca7cc0d739ac44075aad","events":[{"type":"message","message":{"type":"text","id":"616970322256331230","quoteToken":"7k0tRRG6JOS6Dbtk7-dLuZsGHFBnEZIUL6I6W1ej3H7GmtEfqM1vvgwYJggvrB4aw6PnWcsOxt0R6NgtcTN2VcjUciVwZ027bKm2Lq9mbnCgiDRWTP3QwGhv4OjIv5r8GOKvMjM_3fDKQ9qgJCkryw","markAsReadToken":"o17wB9mPc807nBsI7JP4SECMka8nnRkDQBjjfLf3yhZ08FFs-1htmWYGadS3Du6fPPXnNUwRAHztcrFCylYcK2--sOym4GyGrLFBAhVpgvDLLfFlrtf4CY5a08gZRQMRsIgvdLT-A4FI4EvroN_Gfc6prAS_zArXcK6VUSj9fZ2Ws1BXvdm3KBoNeJWF6CBEkAwBjYHragA23BU4fKguNg","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT97YFWVT9W7146AKF4DE830","deliveryContext":{"isRedelivery":false},"timestamp":1780574273000,"source":{"type":"user","userId":"U51821de656e781ae06afc45a00816099"},"replyToken":"c8bdd95808af46ab99ef838e2bf29d8c","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 11:57:55] "POST / HTTP/1.1" 200 -
